In [5]:
import networkx as nx
import numpy as np
import math
import time
import pickle
from ipysigma import Sigma
from composer import * 
#from influence_graph import *

In [7]:
from bs4 import BeautifulSoup
import requests
import re
import networkx as nx
import spacy
import time
import pickle
import json
from composer import *


"""les relations d'influence entre compositeurs sont qualifiées plus précisément qu'avec la seule présence de l'un dans la page wikipedia de l'autre grâce à la fonction orienting with dates"""
"""compléter avec les influences - même peu nombreuses - spécifiées dans wikidata, en plus de celles des élèves - réfléchir jusqu'où aller dans la profondeur des élèves """
"""ajouter une détection des liens vers des pages de compositeurs sur wikipedia"""

def get_person_entities(text):
    nlp = spacy.load("en_core_web_sm")
    doc = nlp(text)
    L=[]
    for ent in doc.ents:
        if ent.label_=='PERSON':
            #print(ent.text, ent.start_char, ent.end_char, ent.label_)
            L.append(ent.text)
    #print(L)
    return L

def get_wikipedia_page_text(name,language="en"):
    url = f"https://{language}.wikipedia.org/w/index.php?search={name}"
    page = requests.get(url)
    soup = BeautifulSoup(page.content, "html.parser")
    text=soup.get_text()
    if language =="en" and (name+" - Wikipedia" in text or name.split(" ")[-1]+" - Wikipedia" in text):
        return get_wikipedia_page_text(name+" (composer)")
    elif language=="fr" and (name+" - Wikipédia" in text or name.split(" ")[-1]+" - Wikipédia" in text):
        return get_wikipedia_page_text(name+" (compositeur)")
    return text

def composer_filter_by_dict(list_of_names, dico):
    """filtre les noms qui sont ceux de compositeurs à partir d'un dictionnaire de compositeurs, évite d'avoir à rechercher dans wikidata à chaque fois donc plus rapide mais moins robuste, à compléter peut-être avec le TD qui utilisait des Trie"""
    list_of_names = list(filter(lambda x: x.strip() in dico, list_of_names))
    return list_of_names

#composer_filter_by_dict(["Richard Wagner","Bach","Mozart"],{"Richard Wagner":(1813,1883),"Johann Sebastian Bach":(1685,1750)})

def composer_filter(list_of_names):
    compo_dico={}
    cpt=0
    for n in list_of_names:
        c_id = find_id(n)
        if c_id is not None and c_id not in compo_dico:
            c = is_composer(c_id,n)
            #print(is_composer(n))
            if c is not None :
                if cpt==0:
                    compo=c
                compo_dico[c_id]=c
                print(n,c_id,f"is the composer number {cpt} found in {compo}'s page")
                cpt+=1
    return compo,compo_dico


def find_compositional_influences(name,fr=False):#problème : l'algo saute souvent Beethoven, y compris chez Wagner
    content = get_wikipedia_page_text(name)
    if fr:
        content+=get_wikipedia_page_text(name,"fr")
    persons = get_person_entities(content)
    return(composer_filter(persons))

def build_list_of_influences(enum_of_composers,french=False):
    list_of_influ_dict=[]
    for composer in enum_of_composers:
        list_of_influ_dict.append(find_compositional_influences(composer,fr=french))
    return list_of_influ_dict



def orienting_with_dates(list_of_dico,compo_dico={}):
    """completing the influences of all encountered compsoers; orienting the influence relations when no doubt exists because of birth-death dates"""

    for compo,dico in list_of_dico:
        cpt=0
        b=False #boolean equal to True iff we have access the the birth date of the first composer
        d=False #boolean equal to True iff we have access the the death date of the first composer
        #print(dico)
        for c in dico.values():

            if c.id not in compo_dico :
                compo_dico[c.id]=c

            if cpt==0: #the first element of each dictionary is the one whose wikipedia page was used to retrieve all the others present in the current dictionary
                #compo = compo_dico[c.id]
                cpt+=1
                print("1st composer of this page :",compo,compo.id,compo_dico[compo.id].influences)
                birth_date_compo = compo.birth
                death_date_compo = compo.death
                if birth_date_compo is not None :
                    print("db:"+str(compo.birth))
                    b = True

                if death_date_compo is not None :
                    print("dd:"+str(compo.death))
                    d = True

            else :
                birth_date = c.birth
                death_date = c.death

                if d and birth_date is not None:
                    if birth_date > death_date_compo-20:#if compo died at most 2O years after the birth of c then we assume only compo could influence c and not the other way around
                        compo_dico[compo.id].influences.add(c)
                        #print(f"added {c} to {compo_dico[compo.id]}")
                    elif b and death_date is not None:
                        if death_date < birth_date_compo+20:#if the current composer c died at most 20 years after the birth of compo we assume compo could not influence c and that only c influenced compo
                            compo_dico[c.id].influences.add(compo)
                            #print(f"added {compo_dico[compo.id]} to {c}")
                        else:
                            compo_dico[compo.id].influences.add(c)
                            #print(f"added {c} to {compo_dico[compo.id]} in the new way")
                            compo_dico[c.id].influences.add(compo)
                            #print(f"added {compo_dico[compo.id]} to {c}")
                    else:
                        compo_dico[compo.id].influences.add(c)
                        #print(f"added {c} to {compo_dico[compo.id]}")
                        compo_dico[c.id].influences.add(compo)
                        #print(f"added {compo_dico[compo.id]} to {c}")

                elif b and death_date is not None:
                        if death_date < birth_date_compo+20:#if the current composer c died at most 20 years after the birth of compo we assume compo could not influence c and that only c influenced compo
                            compo_dico[c.id].influences.add(compo)
                            #print(f"added {compo_dico[compo.id]} to {c}")
                        #elif d and birth_date is not None:
                        #    if birth_date > death_date_compo-20:#if compo died at most 2O years after the birth of c then we assume only compo could influence c and not the other way around
                        #        compo_dico[compo.id].influences.add(c)
                        else:
                            compo_dico[compo.id].influences.add(c)
                            #print(f"added {c} to {compo_dico[compo.id]}")
                            compo_dico[c.id].influences.add(compo)
                            #print(f"added {compo_dico[compo.id]} to {c}")

                else:#we cannot say anything for sure since they were contemporary so by default we assume they influenced each other
                    compo_dico[compo.id].influences.add(c)
                    #print(f"added {c} to {compo_dico[compo.id]}")
                    compo_dico[c.id].influences.add(compo)
                    #print(f"added {compo_dico[compo.id]} to {c}")
        print(compo_dico[compo.id],"has influenced",compo_dico[compo.id].influences)

    return compo_dico



def build_graph_from_dict(compo_dico,G=nx.DiGraph()):
    for compo in compo_dico.values():
        print("in build_graph_from_dict",compo,"has influenced",compo.influences)
        for c in compo.influences:
            G.add_edge(compo,c)
            print(f"added {c} to {compo_dico[compo.id]}")

    # save graph object to file
    pickle.dump(G, open(str(next(iter(compo_dico.keys())))+".pickle", 'wb'))
    return G

def graph_of_influences(list_of_compo_names,fr=True,G=nx.DiGraph()):
    list_of_influ_dicts=build_list_of_influences(list_of_compo_names,fr)#put false if you're only interested in the english wiki
    compo_dico = orienting_with_dates(list_of_influ_dicts)
    return build_graph_from_dict(compo_dico,G)


In [9]:
def prepare_sigma_with_composer_ids(original_graph):
    """Relabels a graph using Composer.id as node labels and returns a Sigma widget."""
    #id_map = {node: int(node.id.replace('Q','')) for node in original_graph.nodes()}
    id_map={node: str(node)+" - "+str(node.id) for node in original_graph.nodes()}
    graph = nx.relabel_nodes(original_graph, id_map)

    # Clean all node/edge attributes to make sure they're JSON serializable
    for n, d in graph.nodes(data=True):
        for k, v in d.items():
            if not isinstance(v, (str, int, float, bool, type(None))):
                d[k] = str(v)

    for u, v, d in graph.edges(data=True):
        for k, val in d.items():
            if not isinstance(val, (str, int, float, bool, type(None))):
                d[k] = str(val)

    layout = {
        node: {
            "x": math.cos(i * 2 * math.pi / len(graph)),
            "y": math.sin(i * 2 * math.pi / len(graph)),
        }
        for i, node in enumerate(graph)
    }

    pagerank_scores = nx.pagerank(graph)

    for k, v in pagerank_scores.items():
        assert isinstance(v, (int, float)), f"Non-numeric pagerank for node {k}: {v}"

    sigma = Sigma(
        graph=graph,
        layout=layout,
        default_edge_type="curve",
        node_border_color_from="node",
        label_font="cursive",
        node_size=pagerank_scores
    )

    return sigma

#prepare_sigma_with_composer_ids(G)
import pickle
from ipysigma import Sigma

def sigma_graph_from_file(filename):
    G = pickle.load(open(filename, 'rb'))
    return G,prepare_sigma_with_composer_ids(G)
    


#G,S = sigma_graph_from_file("Q205519.pickle")
#S
#G=graph_of_influences(["Paul Le Flem","Jean Cras"])
G=pickle.load(open("Q964256.pickle", 'rb'))
S = prepare_sigma_with_composer_ids(G)
S

Sigma(nx.DiGraph with 101 nodes and 152 edges)

In [51]:
"""Louvain"""
"""
Louv=nx.community.louvain.louvain_communities(G,weight='weight')
print("number of communities with Louvain algorithm : ",len(Louv))
print(Louv,"\n")
"""
#K=G.to_undirected(reciprocal=False, as_view=False)
"""k-cliques"""
"""
L=[list(nx.community.k_clique_communities(K,i)) for i in range(2,12)]
lens=list(map(len,L))
print(lens)#number of communities found for each k value between 2 and 11
L=list(nx.community.k_clique_communities(K,5))#for k=5 the maximum number of communities (4) is obtained
print("number of communities with k-cliques algorithm for k = 5 : ", len(L))
print(L,"\n")
"""

"""Girvan-Newman"""
import itertools
"""N=79
comp = nx.community.girvan_newman(K)
for communities in itertools.islice(comp, N):
    print(len(tuple(sorted(c) for c in communities)))"""
k = 4 #number of communities we want to separate
comp = nx.community.girvan_newman(G)
limited = itertools.takewhile(lambda c: len(c) <= k, comp)
i=1
for communities in limited:
    print("number of communities with Girvan-Newman algorithm after", str(i), " iterations : ", len(communities))#tuple(sorted(c) for c in communities)
    i+=1
#print(tuple(sorted(c) for c in communities),"\n")
print(communities)


"""label propagation"""
"""
labcommus=nx.community.label_propagation_communities(G)
print("number of communities with label propagation : ", len(labcommus))
print(labcommus)
"""


number of communities with Girvan-Newman algorithm after 1  iterations :  2
number of communities with Girvan-Newman algorithm after 2  iterations :  3
number of communities with Girvan-Newman algorithm after 3  iterations :  4
({Béla Bartók, Paul Le Flem, Q96985, Dorian Le Gallienne, Edgard Varèse, Stravinsky, Pierre Even, Zoltán Kodály, Edgard Varèse, Joseph Canteloube, Q316280, Paul Le Flem, Jenő Hubay, Georges Taconet, Jean Langlais, Q213661, Q1349386, Joseph Farigoul, Jean Langlais, Stravinsky, ↑ Debussy, Béla Bartók, Joseph Farigoul, Q15973497, Q665460, Georges Taconet, Q1645018, Henri Collet, Q364179, Adrien Rougier, César Thomson, Michel Legrand, Jacques Ibert, Gustav Mahler, Pierre-Yves Moign, Michel Legrand, César Thomson, Siegfried Salomon, Vítězslava Kaprálová, Q50286844, Giorgio Koukl, Q153776, Vítězslava Kaprálová, Paul Ladmirault, Witold Lutoslawski, Jacques Ibert, Siegfried Salomon, Adrien Rougier, Vincent d'Indy, Vincent d'Indy, Q56310080, Q3289087, Q119957, Nikolai My

'\nlabcommus=nx.community.label_propagation_communities(G)\nprint("number of communities with label propagation : ", len(labcommus))\nprint(labcommus)\n'

In [53]:
from composer import * 
g=pickle.load(open("Q964256.pickle", 'rb'))
id_map={node: str(node)+" - "+str(node.id) for node in g.nodes()}
graph = nx.relabel_nodes(g, id_map)

# Clean all node/edge attributes to make sure they're JSON serializable
for n, d in graph.nodes(data=True):
    for k, v in d.items():
        if not isinstance(v, (str, int, float, bool, type(None))):
            d[k] = str(v)

for u, v, d in graph.edges(data=True):
    for k, val in d.items():
        if not isinstance(val, (str, int, float, bool, type(None))):
            d[k] = str(val)

"""layout = {
    node: {
        "x": math.cos(i * 2 * math.pi / len(graph)),
        "y": math.sin(i * 2 * math.pi / len(graph)),
    }
    for i, node in enumerate(graph)
}"""

pagerank_scores = nx.pagerank(graph)

for k, v in pagerank_scores.items():
    assert isinstance(v, (int, float)), f"Non-numeric pagerank for node {k}: {v}"

era_categories = ["Medieval","Baroque", "Classique", "Romantique", "XXe", "contemporain"]
colors = {
    "Medieval": "#654321", #Brown
    "Baroque": "#2ECC71",      # Green
    "Classique": "#F1C40F",    # Yellow
    "Romantique": "#E74C3C",       # Red
    "XXe": "#3498DB",  # Blue
    "contemporain": "#7F00FF", #Lilac
    "Unknown": "#95A5A6"
    
}

partitions = {"Medieval":[],"Baroque":[], "Classique":[], "Romantique":[], "XXe":[], "contemporain":[]}

for node in g.nodes:
    era = node.get_era()  # Default to "medium"
    graph.nodes[str(node)+" - "+str(node.id)]["color_node"] = colors.get(era, "#95A5A6")  # Gray if unknown
    #g.nodes[node]["size"] = 10  # Set a default size
    #partitions[era].append(graph.nodes[str(node)+" - "+str(node.id)])
    partitions[era].append(str(node)+" - "+str(node.id))


for source, target in graph.edges:
    graph.edges[source, target]["color_edge"] = graph.nodes[target]["color_node"]

layout = {}

# Grid positions for each era cluster center
grid_positions = {
    "Medieval": (0, 0),
    "Baroque": (-3, 0),
    "Classique": (0, 2),
    "Romantique": (3, 0),
    "XXe": (-2, -2),
    "contemporain": (2, -2)
}

# Assign circular layouts within each cluster
for era, node_names in partitions.items():
    base_x, base_y = grid_positions.get(era, (0, 0))
    angle_step = 2 * np.pi / max(1, len(node_names))
    radius = 1.5  # Distance from center of cluster

    for i, node_name in enumerate(node_names):
        angle = i * angle_step
        layout[node_name] = {
            "x": base_x + radius * np.cos(angle),
            "y": base_y + radius * np.sin(angle)
        }

# Visualize with ipysigma
sigma = Sigma(
    graph,
    raw_node_color="color_node",
    raw_edge_color="color_edge",
    layout=layout,
    node_label="label",
    default_edge_type="curve",
    node_border_color_from="node",
    label_font="cursive",
    node_size=nx.pagerank(g)
)

sigma

Sigma(nx.DiGraph with 101 nodes and 152 edges)